In [22]:
import pandas as pd
import numpy as np
import holidays

def load_df(url: str):
  df = pd.read_csv(url, parse_dates=['InvoiceDate'])
  df['ds'] = df['InvoiceDate']
  df['y'] = df['TotalSales']
  return df[['ds', 'y']]

def enrich_data(df):
  df['dayofweek'] = df['ds'].dt.dayofweek
  df['weekofyear'] = df['ds'].dt.isocalendar().week
  df['is_month_start'] = df['ds'].dt.is_month_start.astype(int)
  df['is_month_end'] = df['ds'].dt.is_month_end.astype(int)
  df['is_quarter_start'] = df['ds'].dt.is_quarter_start.astype(int)
  df['is_quarter_end'] = df['ds'].dt.is_quarter_end.astype(int)
  df['is_year_start'] = df['ds'].dt.is_year_start.astype(int)
  df['is_year_end'] = df['ds'].dt.is_year_end.astype(int)
  
  uk_holidays = holidays.UK(years=df['ds'].dt.year.unique())
  df['is_holiday'] = df['ds'].isin(uk_holidays).astype(int)
  df['economic_index'] = np.sin(df['ds'].dt.dayofyear/365*2*np.pi) * 0.5 + 0.5
  df['days_since_start'] = (df['ds'] - df['ds'].min()).dt.days
  
  return df

extra_features = ['dayofweek', 'weekofyear', 'is_month_start', 'is_month_end', 
                  'is_quarter_start', 'is_quarter_end', 'is_year_start', 'is_year_end', 
                  'is_holiday', 'economic_index', 'days_since_start']

In [23]:
df_train = load_df('../../data/dataset_training.csv')
df_predict = load_df('../../data/dataset_prediction.csv')

df_train = enrich_data(df_train)
df_predict = enrich_data(df_predict)
df_predict = df_predict.fillna(0)

df_predict.head()

/tmp/ipykernel_20078/3292498376.py:22: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday'] = df['ds'].isin(uk_holidays).astype(int)
/tmp/ipykernel_20078/3292498376.py:22: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday'] = df['ds'].isin(uk_holidays).astype(int)


,ds,y,dayofweek,weekofyear,is_month_start,is_month_end,is_quarter_start,is_quarter_end,is_year_start,is_year_end,is_holiday,economic_index,days_since_start
0,2011-11-09,30888.52,2,45,0,0,0,0,0,0,0,0.109852,0
1,2011-11-10,37581.64,3,45,0,0,0,0,0,0,0,0.115293,1
2,2011-11-11,38162.18,4,45,0,0,0,0,0,0,0,0.120847,2
3,2011-11-13,25752.48,6,45,0,0,0,0,0,0,0,0.132291,4
4,2011-11-14,32750.20,0,46,0,0,0,0,0,0,0,0.138178,5


In [ ]:
from prophet import Prophet

# Configuración del modelo
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
    holidays_prior_scale=10,
    interval_width=0.95
)

model.add_country_holidays(country_name='UK')
model.add_seasonality(name="monthly", period=30.5, fourier_order=10)
model.add_seasonality(name='weekly', period=24*6, fourier_order=5)

# for reg in extra_features:
#     model.add_regressor(reg)

model.fit(df_train)

17:28:39 - cmdstanpy - INFO - Chain [1] start processing
17:28:39 - cmdstanpy - INFO - Chain [1] done processing


In [21]:
columns = ['ds'] + extra_features

forecast_test = model.predict(df_predict[columns])
result = df_predict.merge(forecast_test[['ds', 'yhat']], on='ds', how='left')

# forecast_test = model.predict(df_predict[['ds']])
# result = df_predict.merge(forecast_test[['ds', 'yhat']], on='ds', how='left')

result.head()

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [43]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(result['y'], result['yhat'])
print(f"MAE: {mae}")

mae_relativo = (mae / result['y'].mean()) * 100
print(f"MAE Relativo: {mae_relativo:.2f}%")

MAE: 4747.270193928004
MAE Relativo: 14.58%
